# Exploratory Data Analysis

In [3]:
# Imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import HTML
from ipywidgets import Tab, Output
from IPython.display import display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [6]:
releases_df = pd.read_csv('../data/top_releases.csv')
stock_data = pd.read_csv('../data/stock_data.csv')
returns_df = pd.read_csv('../data/abnormal_returns.csv')
engagement_df = pd.read_csv('../data/reddit_engagement.csv')
minimal_posts_df = pd.read_csv('../data/reddit_engagement_minimal.csv')

all_genres = list(releases_df.columns[9:-2])

clean_stock_data = stock_data.dropna()

# Merge returns_df and engagement_df on the 'title' and 'release_date' columns
merged_data = pd.merge(returns_df, engagement_df, on=['name', 'release_date'], how='inner')
# Select only the relevant columns
merged_data = merged_data.drop(columns=['id', 'first_air_date', 'provider_id'])

genre_counts = merged_data[all_genres].sum().sort_values(ascending=False)

abnormal_returns = returns_df['abnormal_return'].dropna()

merged_data['return_positive'] = (merged_data['abnormal_return'] > 0).astype(int)

In [12]:
# summary = merged_data.describe().reset_index()

# col_widths = [200] + [120] * (len(summary.columns) - 1)


# fig = go.Figure(data=[go.Table(
#     columnwidth=col_widths,
#     header=dict(values=list(summary.columns),
#                 fill_color='lightgrey'),
#     cells=dict(values=[summary[col] for col in summary.columns])
# )])
# fig.update_layout(width=800, height=400,
#                   title='Summary Statistics for the Merged Dataset')
# fig.show()

merged_data_summary = merged_data.describe()
print("Summary Statistics for the Merged Dataset:")
display(merged_data_summary)

# Histograms for the numerical columns in the merged dataset
numerical_cols = [
    'popularity', 'vote_average', 'vote_count', 'stock_return',
    'market_return', 'abnormal_return',
    'avg_sentiment', 'avg_weighted_sentiment', 'engagement_score'
]

fig = make_subplots(
    rows=3, cols=3,
    subplot_titles=[f'{col} Histogram' for col in numerical_cols],
    horizontal_spacing=0.05, vertical_spacing=0.08
)

for i, col in enumerate(numerical_cols):
    r, c = divmod(i, 3)
    row, col_idx = r+1, c+1

    # add histogram
    fig.add_trace(
        go.Histogram(
            x=merged_data[col],
            name=col,
            opacity=0.75,
            showlegend=False
        ),
        row=row, col=col_idx
    )

    # add mean/median lines
    mean_val   = merged_data[col].mean()
    median_val = merged_data[col].median()

    fig.add_vline(
        x=mean_val,
        line=dict(color='red', dash='dash'),
        annotation_text=f'Mean',
        row=row, col=col_idx
    )
    fig.add_vline(
        x=median_val,
        line=dict(color='green', dash='dash'),
        annotation_text=f'Median',
        row=row, col=col_idx
    )

fig.update_layout(
    width=1600,
    height=400,
    title='Summary Statistics for the Merged Dataset',
    margin=dict(l=20, r=20, t=50, b=20)
)

fig.show()

Summary Statistics for the Merged Dataset:


,popularity,vote_average,vote_count,Action & Adventure,Animation,Comedy,Crime,Documentary,Drama,Family,...,abnormal_return,n_texts,avg_sentiment,std_sentiment,min_sentiment,max_sentiment,avg_weighted_sentiment,max_weighted_sentiment,engagement_score,return_positive
count,674.000000,674.000000,674.000000,674.000000,674.000000,674.000000,674.000000,674.0,674.000000,674.000000,...,674.000000,674.000000,402.000000,402.000000,402.000000,402.000000,402.000000,402.000000,402.000000,674.000000
mean,86.432187,7.576312,6829.899110,0.142433,0.142433,0.237389,0.166172,0.0,0.489614,0.109792,...,0.166941,29.768546,0.160945,0.526995,-0.809369,0.929708,1.989392,13.830509,60.627413,0.513353
std,54.921773,0.738511,6998.357379,0.349753,0.349753,0.425798,0.372512,0.0,0.500263,0.312863,...,3.602336,48.717213,0.183941,0.082469,0.239856,0.106989,2.352762,4.321260,68.570619,0.500193
min,30.307000,5.897000,1005.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,-32.215538,0.000000,-0.563358,0.230730,-0.999900,0.361200,-7.060490,1.979586,1.786000,0.000000
25%,43.037400,7.100000,2268.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,-1.109943,0.000000,0.064827,0.492285,-0.968400,0.897700,0.825811,11.248216,14.337000,0.000000
50%,77.385200,7.622000,3593.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,...,0.036463,11.000000,0.164914,0.531898,-0.883400,0.971700,1.923941,14.392607,27.841000,1.000000
75%,110.676900,8.210000,10325.000000,0.000000,0.000000,0.000000,0.000000,0.0,1.000000,0.000000,...,1.060075,38.000000,0.264431,0.569507,-0.757900,0.997200,3.368389,16.759867,96.300000,1.000000
max,451.814700,8.924000,37003.000000,1.000000,1.000000,1.000000,1.000000,0.0,1.000000,1.000000,...,16.866286,281.000000,0.567145,0.715923,0.000000,0.999800,8.098685,21.747903,347.905000,1.000000


The histograms of the continuous variables reveal several key patterns in their distributions. Most variables—such as popularity, vote_count, engagement_score, and the return-based metrics—exhibit strong right-skewness. In contrast, vote_average, avg_sentiment, and avg_weighted_sentiment appear more symmetrically distributed, with avg_sentiment closely resembling a normal distribution centered around a slightly positive value. Return variables (stock_return, market_return, and abnormal_return) cluster tightly around zero. Overall, the distributions highlight a landscape dominated by a few highly impactful data points against a backdrop of more modest, frequent observations.

In [8]:
# Plot count of genres in the dataset
fig = px.bar(
    x=genre_counts.index,
    y=genre_counts.values,
    labels={'x':'Genre', 'y':'Count'},
    title='Genre Frequency in Movie Collection'
)
fig.update_xaxes(tickangle=45, tickmode='array')
fig.update_layout(margin=dict(t=50, b=150))
fig.show()

The bar chart above displays the frequency distribution of genres across a movie collection, with Drama being the most prevalent genre, followed by Adventure and Action.

In [9]:
engagement_metrics = [
    'popularity', 'vote_average', 'vote_count',
    'avg_sentiment', 'avg_weighted_sentiment', 'engagement_score'
]

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[f'Abnormal Return vs {m}' for m in engagement_metrics],
    horizontal_spacing=0.05, vertical_spacing=0.08
)

for i, metric in enumerate(engagement_metrics):
    r, c = divmod(i, 3)
    row, col_idx = r+1, c+1

    fig.add_trace(
        go.Scatter(
            x=merged_data[metric],
            y=merged_data['abnormal_return'],
            mode='markers',
            showlegend=False
        ),
        row=row, col=col_idx
    )
    fig.add_hline(
        y=0,
        line=dict(color='red', dash='dash'),
        row=row, col=col_idx
    )
    fig.update_xaxes(title_text=metric, row=row, col=col_idx)
    fig.update_yaxes(title_text='Abnormal Return (%)', row=row, col=col_idx)

fig.update_layout(
    height=800, width=1200,
    title_text='Correlations between Engagement Metrics and Abnormal Return'
)
fig.show()

The scatterplots display the relationships between abnormal returns and various engagement-related metrics. Overall, the correlations appear weak, with most variables showing no clear linear pattern with abnormal return. \
Engagement indicators such as popularity, vote_average, vote_count, and engagement_score show dispersed patterns with no discernible trend, suggesting limited explanatory power for short-term stock movement. \
Sentiment variables (avg_sentiment and avg_weighted_sentiment) also display low association with abnormal returns, indicating that neither raw sentiment nor weighted sentiment is strongly linked to market reactions in the 3-day window following a release.